> ⚠️ **NOTEBOOK SUPERADO — baseline pixel-level (histórico).**
> Este notebook avalia o modelo no **nível de pixel** (~403k pixels), o que
> **infla as métricas por pseudo-replicação** (AUC ~0,83 aqui vs ~0,5–0,7 no
> nível de amostra). A avaliação canônica e honesta está nos notebooks
> **06 → 07 → 08** (unidade = amostra/ROI, StratifiedKFold k=3, sem leakage,
> com teste de permutação). Mantido apenas como referência do "antes".

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    ConfusionMatrixDisplay, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
from lightgbm import LGBMClassifier

ROOT_DIR = Path("..").resolve()
sys.path.append(str(ROOT_DIR))
sys.path.append(str(ROOT_DIR / "src"))

from src.feature_selectors   import selecionar_bandas_consenso
from src.feature_engineering import build_gold_matrix

print("Imports OK")


In [ ]:
from src.preprocessing import build_preprocessor

PATH_DATA = ROOT_DIR / "data" / "silver level" / "dados_calibrados"
df_raw = pd.read_csv(PATH_DATA / "master_raw.csv")
band_cols = [c for c in df_raw.columns if c not in ("sample_id", "target")]

# Silver guarda só RAW; este baseline usa MSC computado ON-THE-FLY (global,
# pixel-level — coerente com o que este baseline sempre fez antes do refactor).
def _aplicar_pre_global(df, metodo):
    out = df.copy()
    out[band_cols] = build_preprocessor(metodo).fit_transform(df[band_cols].values)
    return out

datasets = {"MSC": _aplicar_pre_global(df_raw, "msc")}

n_pos = int(df_raw.drop_duplicates("sample_id")["target"].sum())
print(f"Dataset (pixel-level): {df_raw['sample_id'].nunique()} amostras "
      f"({n_pos} resist.) | {len(df_raw):,} pixels | {len(band_cols)} bandas")

In [ ]:
# ── Avaliação Baseline: MSC + MI k=15, K-Fold k=3 ───────────────────────────
# Protocolo: StratifiedGroupKFold — grupos por sample_id, sem leakage biológico.
# Seleção MI feita nos pixels de TREINO de cada fold.

METODO_SEL = "mi"
K_BANDAS   = 15
COLUNAS_META = ["sample_id", "target", "block_id", "roi_id", "pixel_id"]

df_msc = datasets["MSC"]
cv     = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
groups = df_msc["sample_id"]
y_full = df_msc["target"]

fold_metricas  = []
fold_info      = []
y_all_true     = []
y_all_pred     = []

print(f"MSC + MI k={K_BANDAS} | K-Fold k=3")
print("="*65)

for fold_idx, (train_idx, test_idx) in enumerate(
        cv.split(df_msc, y_full, groups=groups)):

    df_tr = df_msc.iloc[train_idx].reset_index(drop=True)
    df_te = df_msc.iloc[test_idx].reset_index(drop=True)

    n_tr_px  = len(df_tr)
    n_te_px  = len(df_te)
    n_tr_am  = df_tr["sample_id"].nunique()
    n_te_am  = df_te["sample_id"].nunique()
    n_tr_pos = df_tr.drop_duplicates("sample_id")["target"].sum()
    n_te_pos = df_te.drop_duplicates("sample_id")["target"].sum()

    fold_info.append({
        "Fold": fold_idx + 1,
        "Treino — amostras": f"{n_tr_am} ({int(n_tr_pos)} resist.)",
        "Treino — pixels":   f"{n_tr_px:,}",
        "Teste — amostras":  f"{n_te_am} ({int(n_te_pos)} resist.)",
        "Teste — pixels":    f"{n_te_px:,}",
    })

    print(f"\nFold {fold_idx+1}: treino={n_tr_px:,}px ({n_tr_am} amostras) | "
          f"teste={n_te_px:,}px ({n_te_am} amostras)")

    # Seleção MI nos pixels de TREINO
    bandas = selecionar_bandas_consenso(df_tr, k=K_BANDAS, metodo=METODO_SEL)
    print(f"  Bandas MI selecionadas: {bandas[:5]}...")

    # Gold pixel-level
    gold_tr = build_gold_matrix(df_tr, bandas_selecionadas=bandas,
                                 incluir_features_derivadas=False)
    gold_te = build_gold_matrix(df_te, bandas_selecionadas=bandas,
                                 incluir_features_derivadas=False)

    META = ["sample_id", "target"]
    X_tr = gold_tr.drop(columns=[c for c in META if c in gold_tr.columns])
    y_tr = gold_tr["target"].values
    X_te = gold_te.drop(columns=[c for c in META if c in gold_te.columns])
    y_te = gold_te["target"].values

    model = LGBMClassifier(
        n_estimators=100, learning_rate=0.05, num_leaves=7,
        max_depth=3, min_child_samples=2, lambda_l2=1.0,
        class_weight="balanced", importance_type="gain",
        random_state=42, verbose=-1,
    )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    roc = roc_auc_score(y_te, y_prob) if len(np.unique(y_te)) >= 2 else 0.5
    m = {
        "Fold":      fold_idx + 1,
        "Acurácia":  accuracy_score(y_te, y_pred),
        "Precisão":  precision_score(y_te, y_pred, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "F1-score":  f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC":   roc,
    }
    fold_metricas.append(m)
    y_all_true.extend(y_te)
    y_all_pred.extend(y_pred)
    print(f"  F1={m['F1-score']:.4f} | AUC={m['ROC-AUC']:.4f} | "
          f"Recall={m['Recall']:.4f} | Precisão={m['Precisão']:.4f}")

# ── Tabela de pixels por fold ─────────────────────────────────────────────────
print("\n" + "="*65)
print("PIXELS POR FOLD")
print("="*65)
df_info = pd.DataFrame(fold_info).set_index("Fold")
display(df_info)

# ── Tabela de métricas por fold + média ───────────────────────────────────────
print("\n" + "="*65)
print("MÉTRICAS POR FOLD — MSC + MI k=15, K-Fold k=3")
print("="*65)
df_met = pd.DataFrame(fold_metricas).set_index("Fold")
media_row = df_met.mean().rename("Média")
std_row   = df_met.std().rename("Std")
df_met_full = pd.concat([df_met, media_row.to_frame().T, std_row.to_frame().T])
display(df_met_full.round(4))

y_all_true = np.array(y_all_true)
y_all_pred = np.array(y_all_pred)


In [ ]:
# ── Matriz de confusão acumulada (predições dos 3 folds) ─────────────────────
OUTPUT_DIR = ROOT_DIR / "data" / "gold level" / "features_extraidas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("MSC + MI k=15 — K-Fold k=3 (Baseline)", fontsize=13, fontweight="bold")

# Esquerda: matriz de confusão
cm_val = confusion_matrix(y_all_true, y_all_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=["Sensível (0)", "Resistente (1)"],
)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
for text in axes[0].texts:
    text.set_fontsize(13)
    text.set_fontweight("bold")

f1_acum  = f1_score(y_all_true, y_all_pred, zero_division=0)
rec_acum = recall_score(y_all_true, y_all_pred, zero_division=0)
prec_acum = precision_score(y_all_true, y_all_pred, zero_division=0)

axes[0].set_title(
    f"Matriz de Confusão Acumulada\n"
    f"F1={f1_acum:.3f} | Recall={rec_acum:.3f} | Precisão={prec_acum:.3f}",
    fontsize=10, fontweight="bold"
)
axes[0].set_xlabel("Predito", fontsize=10)
axes[0].set_ylabel("Real", fontsize=10)

# Direita: métricas por fold
metricas_cols = ["Acurácia", "Precisão", "Recall", "F1-score", "ROC-AUC"]
cores_met = ["#1976D2", "#388E3C", "#F57C00", "#C62828", "#6A1B9A"]
x = np.arange(len(metricas_cols))
width = 0.25

for i, m in enumerate(fold_metricas):
    vals = [m[c] for c in metricas_cols]
    axes[1].bar(x + (i - 1) * width, vals, width,
                label=f"Fold {m['Fold']}", alpha=0.85)

# Linha da média
medias = [np.mean([m[c] for m in fold_metricas]) for c in metricas_cols]
axes[1].plot(x, medias, "k--o", linewidth=1.5, markersize=5, label="Média", zorder=5)

axes[1].set_xticks(x)
axes[1].set_xticklabels(metricas_cols, fontsize=9)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("Score", fontsize=10)
axes[1].set_title("Métricas por Fold", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

# Valores da média nas barras
for i, (met, val) in enumerate(zip(metricas_cols, medias)):
    axes[1].text(i, val + 0.03, f"{val:.3f}", ha="center",
                 fontsize=8, fontweight="bold", color="#333")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_msc_mi_kfold.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: baseline_msc_mi_kfold.png")


In [ ]:
# ── Resumo final ─────────────────────────────────────────────────────────────
total_treino_px = sum(int(r["Treino — pixels"].replace(",","")) for r in fold_info)
total_teste_px  = sum(int(r["Teste — pixels"].replace(",","")) for r in fold_info)

print("RESUMO — MSC + MI k=15, K-Fold k=3")
print("="*50)
print(f"Total pixels treino (soma dos folds): {total_treino_px:,}")
print(f"Total pixels teste  (soma dos folds): {total_teste_px:,}")
print(f"Bandas selecionadas (MI):             {K_BANDAS}")
print(f"Features por pixel:                   {K_BANDAS} (só espectros)")
print()
medias = df_met.mean()
for col in ["Acurácia","Precisão","Recall","F1-score","ROC-AUC"]:
    print(f"  {col:12s}: {medias[col]:.4f} ± {df_met[col].std():.4f}")
